```python
from json import loads
import numpy as np

def f():
    global m
    m = {"。": 0}
    for i in "train", "valid":
        for i in open(f"../tmp/news2016zh_{i}.json"):
            i = loads(i)
            for i in i["keywords"], i["desc"], i["title"], i["source"], i["content"]:
                for i in i:
                    yield m.setdefault(i, len(m))
                if "\u4e00" <= i <= "\u9fff":
                    yield 0

a = np.fromiter(f(), "H")
a.tofile("../tmp/news2016zh2.H")
open("../tmp/news2016zh2.txt", "w").write("".join(sorted(m, key=m.__getitem__)))
```

```python
import numpy as np
from multiprocessing.pool import ThreadPool

def f(i):
    c = np.zeros(21482, "I")
    np.add.at(c, a[len(a) * i // 12 : len(a) * (i + 1) // 12], 1)
    return c

a = np.memmap(f"../tmp/news2016zh.H", "H")
with ThreadPool(12) as p:
    p = np.sum(p.map(f, range(12)), 0)

t @ np.log2(t) #-9.655227239498798
    
t = p > p.max() // 100
m = np.zeros(21482, "H")
m[t] = range(t.sum())
m[a[t[a]]].tofile("../tmp/news2016zh100.H")
open("../tmp/news2016zh100.txt", "w").write(np.array([open("../tmp/news2016zh.txt").read()]).view("U1")[t].view(f"U{t.sum()}")[0])

t = p > p.max() // 1000
m = np.zeros(21482, "H")
m[t] = range(t.sum())
m[a[t[a]]].tofile("../tmp/news2016zh1000.H")
open("../tmp/news2016zh1000.txt", "w").write(np.array([open("../tmp/news2016zh.txt").read()]).view("U1")[t].view(f"U{t.sum()}")[0])
```

```python
print(np.array([open("../tmp/news2016zh100.txt").read()]).view("U1")[np.fromfile('../tmp/news2016zh100.H','H',10000)].view('U10000')[0])
```

In [ ]:
from fastai.vision.all import *
from torch.nn import *


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zh100.H", "h"))

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w + 1, w], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.a.size(1) * s.b * s.n)

    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x, x.flatten()


class View(Module):
    def __init__(s, k):
        super().__init__()
        s.k = k

    def forward(s, x):
        return x.view(-1, s.k)

In [ ]:
m = Learner(
    DataLoaders(Dat(256, 256*5, 20), Dat(256, 256*5, 2000)),
    Sequential(
        Embedding(560, 256),
        *[Sequential(View(512), Linear(512, 256, 0), BatchNorm1d(256, track_running_stats=0), Swish()) for _ in range(8)],
        Sequential(Linear(256, 256, 0), BatchNorm1d(256, track_running_stats=0), Swish(), Linear(256, 256)),
        *[Sequential(BatchNorm1d(256, track_running_stats=0), Swish(), Linear(256, 512, 0), View(256)) for _ in range(8)],
        Linear(256, 256, 0),
        BatchNorm1d(256, track_running_stats=0),
        Swish(),
        Linear(256, 560),
    ).cuda(),
    CrossEntropyLoss(),
    metrics=accuracy,
    cbs=[ShowGraphCallback, SaveModelCallback(fname='model3', every_epoch=1, with_opt=1, reset_on_fit=0)],
)
m.lr_find()

In [ ]:
m.fine_tune(40)